In [ ]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [ ]:
import logging
import warnings
warnings.filterwarnings('ignore')

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

注意：运行下面的代码首先要执行 [gen_demo_factor_data.py](../tools/gen_demo_factor_data.py) 脚本生成示例数据。

# HDF5DB

HDF5DB 是基于文件构建的因子库，其和文件系统的对应关系：
* 设定的一个主目录对应于因子库
* 主目录下的每个子目录对应于因子表
* 子目录下的每个 HDF5 文件对应于单个因子

```mermaid
graph TD
    subgraph 逻辑层
        A[因子库]
        B1[因子表]
        B2[因子表]
        A --> B1
        A --> B2
        F1[因子]
        F2[因子]
        B1 --> F1
        B1 --> F2
    end

    subgraph 存储层
        C[主目录]
        D1[子目录]
        D2[子目录]
        C --> D1
        C --> D2
        E1[HDF5文件]
        E2[HDF5文件]
        D2 --> E1
        D2 --> E2
    end

    subgraph Group
        DS1[时点 Dataset]
        DS2[证券代码 Dataset]
        DS3[数据 Dataset]
    end

    A -.-> C
    B1 -.-> D2
    F1 -.-> E2
    E2 -.-> Group

    style A fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style C fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style DS1 fill:#ff8c00,color:#fff,stroke:#cc7000,stroke-width:2px
    style DS2 fill:#ff8c00,color:#fff,stroke:#cc7000,stroke-width:2px
    style DS3 fill:#ff8c00,color:#fff,stroke:#cc7000,stroke-width:2px
```

In [6]:
# 创建因子库对象并 connect
from QuantStudio.Factor.HDF5DB import HDF5DB

FDB = HDF5DB(args={"MainDir": "../data/HDF5"}).connect()
print(qs_help(FDB))

类型: HDF5DB
模块: QuantStudio.Factor.HDF5DB
QS 对象类型: 因子库
QS 对象名称: HDF5DB
QSID: fc259a026745e2f52c8e4fa147af71318890ed298387f54bfdd928b492f374ad
参数集:
    * Name(名称): <class 'str'>, 默认值 'HDF5DB', 当前取值: 'HDF5DB'
    * MainDir(主目录): <class 'pathlib.Path'>, 无默认值, 存放数据的主目录, 当前取值: ..\data\HDF5
    * FileOpenRetryNum(文件打开重试次数): typing.Union[int, float], 默认值 inf, 打开数据文件错误时的重试次数, 当前取值: inf
说明文档:
    基于 HDF5 文件的因子库
    主目录下的每个文件夹表示一张因子表, 每个文件夹下扩展名为 hdf5 的 [HDF5 文件](https://www.hdfgroup.org/) 存储了一个因子的数据。
    每个 HDF5 因子文件有三个 Dataset:
        * ID: 存储因子的 ID 序列数据, shape=(None,), dtype=String, 编码为 utf-8。
        * DateTime: 存储因子的时点序列数据, shape=(None,), dtype=float, 时点转换成 timestamp 存储。
        * Data: 存储因子数据, shape=(None, None), 行数等于 DateTime 的长度, 列数等于 ID 的长度。double 类型的因子数据存储为 float64 类型, string 类型的因子数据存储为 String 类型(编码为 utf-8), object 类型的因子数据存储为 vlen_dtype(np.uint8) 类型。
    因子表的元信息存储在表文件夹下的特殊文件 _TableInfo.h5 中, 没有该文件说明还未写入过元信息
    因子的元数据存储在 HDF5 文件 root group 的 attrs 中
    锁目录下的 LockFile 文件为库锁，修改因子库、因子表以及因

In [7]:
# 获取因子库中的因子表列表
print(FDB.TableNames)

['index_cn_day_bar', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']


# 因子表

In [39]:
# 获取因子表对象
FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})
print(qs_help(FT))

类型: HDF5FactorTable
模块: QuantStudio.Factor.HDF5DB
QS 对象类型: 计算节点-因子表
QS 对象名称: stock_cn_day_bar
QSID: 525da15ba1a99c24a0247eff148ea31df1151a6f2b1c7db93f553468e6d516a6
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'stock_cn_day_bar'
    * Parallel(并行计算): <class 'bool'>, 默认值 True, 当前取值: True
    * TaskExecutor(并行执行器): typing.Optional[concurrent.futures._base.Executor], 默认值 None, 给到节点用于并行计算, 当前取值: None
    * LookBack(回溯天数): typing.Union[int, float], 默认值 0, 缺失填充回溯的天数, 当前取值: 0
    * OnlyStartLookBack(只起始日回溯): <class 'bool'>, 默认值 False, 如果为 True, 表示只对提取数据的第一个时点进行缺失填充, 之后的时点不填充, 当前取值: False
    * OnlyLookBackNontarget(只回溯非目标日): <class 'bool'>, 默认值 False, 如果为 True, 表示只用不在提取时点序列中的数据进行缺失填充, 当前取值: False
    * OnlyLookBackDT(只回溯时点): <class 'bool'>, 默认值 False, 如果为 True, 表示所有 ID 统一沿着时点字段进行回溯填充, 不单独填充, 当前取值: False
    * TargetDT(目标时点): typing.Optional[datetime.datetime], 默认值 None, 非 None 表示只取该时点的值返回, 当前取值: None
说明文档:
    HDF5DB 库中因子表


In [40]:
# 获取因子表中的因子列表
print(FT.FactorNames)

['amount', 'close', 'high', 'low', 'open', 'volume']


## 元信息

In [41]:
# 获取因子表元信息的方法说明
print(qs_help(FT.getMetaData))

类型: method (bound to HDF5FactorTable)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5FactorTable.getMetaData(key: Optional[str] = None) -> Union[Any, pandas.core.series.Series]
说明文档:
    获取因子表的元信息, 元信息由若干个键值对组成
    
    Args:
        key: 元信息键, None 表示获取所有的元信息
    
    Returns:
        如果 key 非 None 则返回该 key 对应的元信息
        如果 key=None, 则返回 Series(index=[所有的 key])


In [12]:
# 获取因子表的所有元信息
print(FT.getMetaData())

Series([], dtype: object)


In [13]:
# 获取因子表的某个元信息
print(FT.getMetaData(key="Description"))

None


## 时点序列

In [42]:
# 获取时点序列的方法
print(qs_help(FT.getDateTime))

类型: method (bound to HDF5FactorTable)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5FactorTable.getDateTime(ifactor_name: Optional[str] = None, iid: Optional[str] = None, start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None) -> List[datetime.datetime]
说明文档:
    获取时点序列
    
    Args:
        ifactor_name: 给定的因子名称, 非 None 表示获取该因子的时点序列, None 表示获取表的时点序列
        iid: 给定的 ID, 非 None 表示获取该 ID 的时点序列, None 表示获取所有的时点序列
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该因子表没有固定的时点序列或者无法获取


In [43]:
# 给定起始时点和截止时点, 获取因子表的时点序列
FT.getDateTime(start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5))

[datetime.datetime(2025, 3, 1, 0, 0),
 datetime.datetime(2025, 3, 2, 0, 0),
 datetime.datetime(2025, 3, 3, 0, 0),
 datetime.datetime(2025, 3, 4, 0, 0),
 datetime.datetime(2025, 3, 5, 0, 0)]

## ID 序列

In [44]:
# 获取因子表 ID 序列的方法
print(qs_help(FT.getID))

类型: method (bound to HDF5FactorTable)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5FactorTable.getID(ifactor_name: Optional[str] = None, idt: Optional[datetime.datetime] = None) -> List[str]
说明文档:
    获取 ID 序列
    
    Args:
        ifactor_name: 给定的因子名称, 非 None 表示获取该因子的 ID 序列, None 表示获取表的 ID 序列
        idt: 给定的时点, 非 None 表示获取该时点的 ID 序列, None 表示获取所有的 ID 序列
    
    Returns:
        ID 序列, 若为空 list, 表示该因子表没有固定的 ID 序列或者无法获取


In [45]:
# 获取因子表的 ID 序列
IDs = FT.getID()
print(IDs[0], ", ..., ", IDs[-1], f"共 {len(IDs)} 个")

000001.SZ , ...,  000020.SZ 共 20 个


## 读取数据

In [15]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs)
print("因子表数据")
print(Data)

因子表数据
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 2 (minor_axis)
Items axis: open to close
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000002.SZ


# 因子

In [16]:
# 获取因子对象
F = FT.getFactor("close")
print(qs_help(F))

类型: Factor
模块: QuantStudio.Factor.Factor
QS 对象类型: 计算节点-因子
QS 对象名称: close
QSID: e26b6dc27f1c7d622be23c89aa75a7667cd2ff71e7a09781d3ccf94abc2c9d6c
参数集:
    * Name(名称): <class 'str'>, 默认值 'Factor', 当前取值: 'close'
    * Parallel(并行计算): <class 'bool'>, 默认值 True, 当前取值: True
    * TaskExecutor(并行执行器): typing.Optional[concurrent.futures._base.Executor], 默认值 None, 给到节点用于并行计算, 当前取值: None
    * Meta(元信息): <class 'dict'>, 默认值 {}, 当前取值: {}
    * SectionIDs(截面ID): typing.Optional[typing.List[str]], 默认值 None, 当前取值: None
    * CalcDTRuler(计算时点标尺): typing.Optional[typing.List[datetime.datetime]], 默认值 None, 当前取值: None
    * CacheEnabled(启用缓存): <class 'bool'>, 默认值 True, 当前取值: True
说明文档:
    因子对象
    因子可看做 DataFrame(index=[时点], columns=[ID])
    时点数据类型是 datetime, ID 的数据类型是 str


## 元信息

In [19]:
# 获取因子元信息的方法
print(qs_help(F.getMetaData))

类型: method (bound to Factor)
模块: QuantStudio.Factor.Factor
签名: Factor.getMetaData(key: Optional[str] = None) -> Union[Any, pandas.core.series.Series]
说明文档:
    获取因子的元信息, 元信息由若干个键值对组成
    
    Args:
        key: 元信息键, None 表示获取所有的元信息
    
    Returns:
        如果 key 非 None 则返回该 key 对应的元信息
        如果 key=None, 则返回 Series(index=[所有的 key])


In [17]:
# 获取因子的所有元信息
print(F.getMetaData(key=None))

DataType    double
dtype: object


In [18]:
# 获取因子的某个元信息
print(F.getMetaData(key="DataType"))

double


## 时点序列

In [46]:
# 获取因子时点序列的方法
print(qs_help(F.getDateTime))

类型: method (bound to Factor)
模块: QuantStudio.Factor.Factor
签名: Factor.getDateTime(iid: Optional[str] = None, start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None, **kwargs) -> List[datetime.datetime]
说明文档:
    获取时点序列
    
    Args:
        iid: 给定的 ID, 非 None 表示获取该 ID 的时点序列, None 表示获取所有的时点序列
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该因子没有固定的时点序列或者无法获取


In [47]:
# 给定起始时点和截止时点, 获取因子的时点序列
F.getDateTime(start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5))

[datetime.datetime(2025, 3, 1, 0, 0),
 datetime.datetime(2025, 3, 2, 0, 0),
 datetime.datetime(2025, 3, 3, 0, 0),
 datetime.datetime(2025, 3, 4, 0, 0),
 datetime.datetime(2025, 3, 5, 0, 0)]

## ID 序列

In [48]:
# 获取因子 ID 序列的方法
print(qs_help(F.getID))

类型: method (bound to Factor)
模块: QuantStudio.Factor.Factor
签名: Factor.getID(idt: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    获取 ID 序列
    
    Args:
        idt: 给定的时点, 非 None 表示获取该时点的 ID 序列, None 表示获取所有的 ID 序列
    
    Returns:
        ID 序列, 若为空 list, 表示该因子没有固定的 ID 序列或者无法获取


In [49]:
# 获取因子的 ID 序列
IDs = F.getID()
print(IDs[0], ", ..., ", IDs[-1])

000001.SZ , ...,  000020.SZ


## 读取数据

In [21]:
# 因子读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = F.readData(ids=IDs, dts=DTs)
print(Data)

            000001.SZ  000002.SZ
2025-01-01   8.115185   4.760840
2025-01-02   0.352198   1.806606
2025-01-03   6.019437   0.633690
2025-01-04   3.936297   3.755494
2025-01-05   4.884425   1.342670


# 因子库的变更操作

HDF5DB 是支持写入和变更的本地因子库（继承自 `QuantStudio.Factor.FactorDB.WritableFactorDB` 的因子库才有以下方法）

## 数据写入

In [ ]:
# 因子库数据写入方法
print(qs_help(FDB.writeData))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.writeData(data: QuantStudio.Core.QSObject.Panel, table_name: str, if_exists: Literal['update', 'replace', 'append'] = 'update', data_type: Dict[str, Literal['double', 'string', 'object']] = {}, **kwargs)
说明文档:
    写入数据
    
    Args:
        data: 待写入的因子数据, Panel(items=[因子], major_axis=[时点], minor_axis=[ID])
        table_name: 因子表名称
        if_exists: 如果该因子已经存在时数据写入的方式, update 表示用新数据更新原数据, append 表示不更新原数据而只增加原来没有的数据, replace 表示完全用新数据替换原数据, 等同于先删除原数据再写入
        data_type: 待写入因子的数据类型, {因子名称: "double" or "string" or "object"}, 如果 data_type 未指定某个因子的数据类型，则交由系统判定


In [ ]:
# 数据写入
from QuantStudio.Core.QSObject import Panel

IDs = [str(i).zfill(6) + ".SZ" for i in range(1, 4)]
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]

Data = Panel({
    "Factor1": pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs),
    "Factor2": pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs)
})
print("待写入的数据 : ")
print(Data)

print("-" * 10)
TargetTable = "TestTable"
FDB.writeData(data=Data, table_name=TargetTable, if_exists="update", data_type={"Factor1":"double", "Factor2":"double"})
print("写入后的因子表列表 : ")
print(FDB.TableNames)

print("-" * 10)
print("写入后的因子列表 : ")
print(FDB.getTable(TargetTable).FactorNames)

待写入的数据 : 
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 3 (minor_axis)
Items axis: Factor1 to Factor2
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000003.SZ
----------
写入后的因子表列表 : 
['TestTable', 'index_cn_day_bar', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']
----------
写入后的因子列表 : 
['Factor1', 'Factor2']


## 设置因子的元信息

In [24]:
# 设置因子元信息的方法
print(qs_help(FDB.setFactorMetaData))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.setFactorMetaData(table_name: str, ifactor_name: str, key: Optional[str] = None, value: Any = None, meta_data: Optional[dict] = None)
说明文档:
    设置因子的元信息, 元信息由若干个键值对组成
    
    Args:
        table_name: 因子表名称
        ifactor_name: 因子名称
        key: 元信息键
        value: 元信息值
        meta_data: 若干组键值对元信息


In [25]:
# 设置因子的元信息
TargetTable = "TestTable"
TargetFactor = "Factor1"
FT = FDB.getTable(TargetTable)
print("设置前的元信息 : ")
print(FT.getFactorMetaData(factor_names=[TargetFactor]))

print("-" * 10)
FDB.setFactorMetaData(table_name=TargetTable, ifactor_name=TargetFactor, key="Description", value="这是一个测试因子")
print("设置后的元信息 : ")
print(FT.getFactorMetaData(factor_names=[TargetFactor]))

设置前的元信息 : 
        DataType
Factor1   double
----------
设置后的元信息 : 
        DataType Description
Factor1   double    这是一个测试因子


## 重命名因子

In [26]:
# 重命名因子的方法
print(qs_help(FDB.renameFactor))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.renameFactor(table_name: str, old_factor_name: str, new_factor_name: str)
说明文档:
    对给定表中的因子重命名
    
    Args:
        table_name: 因子表名称
        old_factor_name: 原因子名
        new_factor_name: 新因子名


In [27]:
# 重命名因子
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print("重命名前因子列表 : ")
print(FT.FactorNames)

print("-" * 10)
FDB.renameFactor(table_name=TargetTable, old_factor_name="Factor1", new_factor_name="NewFactor1")
print("重命名后因子列表 : ")
print(FT.FactorNames)

重命名前因子列表 : 
['Factor1', 'Factor2']
----------
重命名后因子列表 : 
['Factor2', 'NewFactor1']


## 删除因子

In [28]:
# 删除因子的方法
print(qs_help(FDB.deleteFactor))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.deleteFactor(table_name: str, factor_names: List[str])
说明文档:
    删除给定表中的某些因子
    
    Args:
        table_name: 因子表名称
        factor_names: 待删除的因子名列表


In [29]:
# 删除因子
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print("删除前因子列表 : ")
print(FT.FactorNames)

print("-" * 10)
FDB.deleteFactor(table_name=TargetTable, factor_names=["NewFactor1"])
print("删除后因子列表 : ")
print(FT.FactorNames)

删除前因子列表 : 
['Factor2', 'NewFactor1']
----------
删除后因子列表 : 
['Factor2']


## 设置表的元信息

In [30]:
# 设置因子表元信息的方法
print(qs_help(FDB.setTableMetaData))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.setTableMetaData(table_name: str, key: Optional[str] = None, value: Any = None, meta_data: Optional[dict] = None)
说明文档:
    设置因子表的元信息, 元信息由若干个键值对组成
    
    Args:
        table_name: 因子表名称
        key: 元信息键
        value: 元信息值
        meta_data: 若干组键值对元信息


In [31]:
# 设置表的元信息
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print("设置前的元信息 : ")
print(FT.getMetaData())

print("-" * 10)
FDB.setTableMetaData(table_name=TargetTable, key="Description", value="这是一张测试表")
print("设置后的元信息 : ")
print(FT.getMetaData())

设置前的元信息 : 
Series([], dtype: object)
----------
设置后的元信息 : 
Description    这是一张测试表
dtype: object


In [32]:
# 设置表的元信息
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print("设置前的元信息 : ")
print(FT.getMetaData())

print("-" * 10)
FDB.setTableMetaData(table_name=TargetTable, meta_data={"Description": "这还是一张测试表", "aha": 123})
print("设置后的元信息 : ")
print(FT.getMetaData())

设置前的元信息 : 
Description    这是一张测试表
dtype: object
----------
设置后的元信息 : 
Description    这还是一张测试表
aha                 123
dtype: object


## 重命名表

In [33]:
# 重命名表的方法
print(qs_help(FDB.renameTable))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.renameTable(old_table_name: str, new_table_name: str)
说明文档:
    重命名表
    
    Args:
        old_table_name: 原表名
        new_table_name: 新表名


In [34]:
# 重命名表
print("重命名前因子表 : ")
print(FDB.TableNames)

print("-" * 10)
FDB.renameTable(old_table_name="TestTable", new_table_name="NewTestTable")
print("重命名后因子表 : ")
print(FDB.TableNames)

重命名前因子表 : 
['TestTable', 'index_cn_day_bar', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']
----------
重命名后因子表 : 
['NewTestTable', 'index_cn_day_bar', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']


## 删除表

In [35]:
# 删除表的方法
print(qs_help(FDB.deleteTable))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.deleteTable(table_name: str)
说明文档:
    删除表
    
    Args:
        table_name: 表名


In [36]:
# 删除表
print("删除前因子表 : ")
print(FDB.TableNames)

print("-" * 10)
FDB.deleteTable(table_name="NewTestTable")
print("删除后因子表 : ")
print(FDB.TableNames)

删除前因子表 : 
['NewTestTable', 'index_cn_day_bar', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']
----------
删除后因子表 : 
['index_cn_day_bar', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']
